## 0. Load

In [1]:
#Libraries
import os
import sys
import dask
import pandas as pd
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

## 1. Generate cloud top data

In [ ]:

#--------------------------------------------------------------
#Helper functions 
#--------------------------------------------------------------

def time_filter(data):
    winter_month = list([1,2,11,12])
    time_months = data['time'].dt.month
    time_mask = time_months.isin(winter_month)
    data = data.where(time_mask, drop=True)
    return data

def lat_lon_filter(data):
    lat_mask = ((data['lat'] >= 30) & (data['lat'] <= 80)) | ((data['lat'] >= -80) & (data['lat'] <= -30))
    lon_mask = (data['lon'] >= -80) | (data['lon'] <= 80)
    data = data.where(lat_mask & lon_mask, drop=True)
    return data

In [6]:
#--------------------------------------------------------------
#Input data 
#--------------------------------------------------------------
#Folder path
folder_path = "/wolke_scratch/dnikolo/ICON_Data/"
array_path = "/wolke_scratch/dnikolo/ICON_Data/Cloud_top_data_sample/"
file = str(sys.argv[1])
file_path="/wolke_scratch/dnikolo/ICON_Data/atm_3d_remap_test.nc"
# file_path = folder_path + str(file)

In [7]:

from datetime import datetime, timedelta

def float_time_to_datetime(t):
    """
    Convert 'day as %Y%m%d.%f' float (e.g. 20080905.010417)
    to a Python datetime.
    """
    t = float(t)
    date_int = int(t)
    frac = t - date_int  # fraction of day

    year  = date_int // 10000
    month = (date_int % 10000) // 100
    day   = date_int % 100

    # fraction of day -> seconds (round to nearest second)
    seconds = int(round(frac * 24 * 3600))

    return datetime(year, month, day) + timedelta(seconds=seconds)

In [ ]:
#---------------------------------------------------------------
#Temperature range loop
#---------------------------------------------------------------
T_ranges = [(i,i+6) for i in range(273-6, 273-38, -6)]
folder_path="/wolke_scratch/dnikolo/ICON_Data/3d_sample/"
for file in os.listdir(folder_path):
    # if file.startswith("test01_atm_3d"):

    # if file.startswith("test01_atm_3d_ml_20080901"):
    if file.startswith("test01_atm_3d"):
        file_path = folder_path + str(file)
    else:
        continue
    #---------------------------------------------------------------
    #Data setup
    #---------------------------------------------------------------
    #Load data with Dask
    data = xr.open_dataset(
        file_path,
        chunks={'time': 1}
    )
    data = lat_lon_filter(data)
    #Load variables
    CLDLIQ = data['clw']
    CLDICE = data['cli']
    TWC = CLDLIQ + CLDICE
    T = data['ta']
    TWC_threshold = 0.00001

    CC_3d_mask = TWC > TWC_threshold
    CC_3d_mask_index = CC_3d_mask.argmax(dim='height')

    #Remove values with no cloud 

    CC_3d_check = ~CC_3d_mask.any(dim='height')
    CC_3d_final = xr.where(CC_3d_check, -999, CC_3d_mask_index)
    ratio = xr.where(CLDLIQ != 0, CLDICE / CLDLIQ, 0)
    CPH = xr.where(CC_3d_mask,ratio +1,0)
    #b and a / b or 0
    CTT_results=[]
    CPH_results=[]


    for T_range in T_ranges:
        #Temperature bounds
        T_min, T_max = T_range
        #Mask cloudy gridboxes within T range
        CTT_mask = (TWC > TWC_threshold) & (T >= T_min) & (T < T_max)
        #Find lowest level where condition is met (highest since reversed)
        CTT_index = CTT_mask.argmax(dim='height')

        #If gridbox has a cloud + highest cloudy point is within temperature range -> cloud has right CTT
        CTT_filter = (CTT_index == CC_3d_final)

        #Apply mask
        CTT_results.append(T.where(CTT_filter).max(dim='height'))
        CPH_results.append(CPH.where(CTT_filter).max(dim='height'))
    
    with ProgressBar(dt=15.0):
        (
            CTT_output,
            CPH_output

        ) = dask.compute(
            CTT_results,
            CPH_results
            
        )
    
    base_dir = "/wolke_scratch/dnikolo/ICON_Data/Cloud_top_data_sample/Filtered_Data/np"
    for i, cph_temp_range in enumerate(CPH_output):
        # Assign attributes to DataArray otherwise it can not be converted to Dataset
        cph_temp_range = cph_temp_range.assign_attrs({"name": "cph","long_name":"Cloud Phase","units":"unitless"}).rename("cph_filtered")
        
        # convert times to datetimes
        # --- convert float time (YYYYMMDD + frac day) -> datetime64 ---
        t = cph_temp_range["time"].values.astype(float)
        ymd  = np.floor(t).astype(int)
        frac = t - ymd
        base = pd.to_datetime(ymd.astype(str), format="%Y%m%d")
        # Concvert to datetime64[ns] and add 30 seconds to avoid rounding issues when converting to minutes. Minute precision rounds down.
        dt = ((base + pd.to_timedelta(frac, unit="D")).to_numpy(dtype="datetime64[ns]") + np.timedelta64(30, 's')).astype('datetime64[m]')
        
        # Assign value for the time coordinate
        cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))

        # group by time: one dataset per time step
        times, dataset_list = zip(*cph_temp_range.to_dataset().groupby("time",squeeze=False))
        new_data_list = []
        for data,dt in zip(dataset_list,times):
            new_data_list.append(data.assign_coords(time=("time", [dt])))
            data = new_data_list[-1]
            data['time']= data['time'].assign_attrs({'standard_name': 'time',
                                                                        'axis': 'T'})
            data.time.encoding = {
                "units": "seconds since 2008-08-30 00:00:00",
                "calendar": "proleptic_gregorian",
            }
            data['lat']= data['lat'].assign_attrs({'_FillValue': np.nan ,
                                                                    'standard_name': 'latitude',
                                                                    'long_name': 'latitude',
                                                                    'units': 'degrees_north',
                                                                    'axis': 'Y'})
            data['lon']= data['lon'].assign_attrs({'_FillValue': np.nan ,
                                                                    'standard_name': 'longitude',
                                                                    'long_name': 'longitude',
                                                                    'units': 'degrees_east',
                                                                    'axis': 'X'})
        
        # ensure each dataset has `time` as a dimension (len=1), not a scalar coord
        dts = [t for t in times]

        # build one path per dataset in the desired format:
        # 2008/02/01/R_Agg_20_T_06_00_20080201000000.nc
        paths = []
        for dt in dts:
            ts = pd.Timestamp(dt)        # dt is numpy.datetime64
            subdir = ts.strftime("%Y/%m/%d")
            filename = f"R_Agg_30_T_{(273-T_ranges[i][0]):02}_{(273-T_ranges[i][1]):02}_{ts.strftime('%Y%m%d%H%M%S')}.nc"
            full_path = os.path.join(base_dir, subdir, filename)

            # make sure directory exists
            os.makedirs(os.path.dirname(full_path), exist_ok=True)
            paths.append(full_path)
        xr.save_mfdataset(new_data_list, paths)
        #Compute averages
    # CRE_mean = CRE_CTT.mean(dim=('time', 'lat', 'lon'), skipna=True)
    # CRE_T_loop.append(CRE_mean)

    # Compute cloud cover lazily
    # cloudy_gp = CRE_CTT.notnull().sum(dim=('time', 'lat', 'lon'))
    # total_gp = CRE.notnull().sum(dim=('time', 'lat', 'lon'))
    # cloud_fraction = cloudy_gp / total_gp
# The code runs into runtime warning due to dask computation of this line  ratio = xr.where(CLDLIQ != 0, CLDICE / CLDLIQ, 0)
# since the division is executed no matter if the condition is true or false and later maked. The results are correct.


[########################################] | 100% Completed | 15.02 s


/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

[########################################] | 100% Completed | 15.05 s


/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

[########################################] | 100% Completed | 15.03 s


...

/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

[########################################] | 100% Completed | 15.05 s


/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

[########################################] | 100% Completed | 15.00 s


/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
/tmp/ipykernel_39413/2481064219.py:84: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converti

## 2. Test

In [ ]:
for i, cph_temp_range in enumerate(CPH_output):
    # group by time: one dataset per time step
    cph_temp_range = cph_temp_range.assign_attrs({"name": "cph","long_name":"Cloud Phase","units":"unitless"}).rename("cph_filtered")
    # convert times to datetimes
    # --- convert float time (YYYYMMDD + frac day) -> datetime64 ---
    t = cph_temp_range["time"].values.astype(float)
    ymd  = np.floor(t).astype(int)
    frac = t - ymd

    base = pd.to_datetime(ymd.astype(str), format="%Y%m%d")
    dt = ((base + pd.to_timedelta(frac, unit="D")).to_numpy(dtype="datetime64[ns]") + np.timedelta64(30, 's')).astype('datetime64[m]')
    
    cph_temp_range = cph_temp_range.assign_coords(time=("time", dt))
    times, dataset_list = zip(*cph_temp_range.to_dataset().groupby("time",squeeze=False))
    new_data_list = []
    for data,dt in zip(dataset_list,times):
        new_data_list.append(data.assign_coords(time=("time", [dt])))
        data = new_data_list[-1]
        data['time']= data['time'].assign_attrs({'standard_name': 'time',
                                                                    'axis': 'T'})
        data.time.encoding = {
            "units": "seconds since 2008-08-30 00:00:00",
            "calendar": "proleptic_gregorian",
        }
        data['lat']= data['lat'].assign_attrs({'_FillValue': np.nan ,
                                                                'standard_name': 'latitude',
                                                                'long_name': 'latitude',
                                                                'units': 'degrees_north',
                                                                'axis': 'Y'})
        data['lon']= data['lon'].assign_attrs({'_FillValue': np.nan ,
                                                                'standard_name': 'longitude',
                                                                'long_name': 'longitude',
                                                                'units': 'degrees_east',
                                                                'axis': 'X'})